In [1]:
!pip cache purge

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# install library untuk efisiensi memori (xformers, bitsandbytes) dan modul training LLM (trl, peft, accelerate)
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

print("[INFO] semua library berhasil diinstal!")

Files removed: 0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-u6s_kw1y/unsloth_38beab6bbc484560ba7f6266f749d33e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-u6s_kw1y/unsloth_38beab6bbc484560ba7f6266f749d33e
  Resolved https://github.com/unslothai/unsloth.git to commit 8ca09b86dc5085880cb0d74a14b0df09d468d8ac
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.4 MB

In [2]:
# load base model & Tokenizer (4-Bit)
from unsloth import FastLanguageModel
import torch

# konfigurasi batas maksimal token dan efisiensi memori
max_seq_length = 512
dtype = None
load_in_4bit = True

# model dan tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("[INFO] Model dan Tokenizer berhasil dimuat!")
print("Token EOS (penanda akhir kalimat) model ini adalah:", tokenizer.eos_token)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


[INFO] Model dan Tokenizer berhasil dimuat!
Token EOS (penanda akhir kalimat) model ini adalah: <|eot_id|>


In [3]:
# dataset mapping
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# dataset bahasa indonesia
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

# sinkronasi format chat dengan tokenizer model Llama-3
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

# fungsi untuk merakit kolom mentah menjadi format chat
def format_alpaca(examples):
    texts = []
    for user_prompt, output in zip(examples["input"], examples["output"]):
        # susun peran (role)
        messages = [
            {"role": "user", "content": user_prompt.strip()},
            {"role": "assistant", "content": output.strip()},
        ]

        # template dan token spesial (tanpa menambahkan prompt generatif baru)
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)

    return {"text": texts}

# mapping ke seluruh baris data
mapped_dataset = dataset.map(format_alpaca, batched=True)

print("[INFO] Dataset berhasil diformat!")
print("\n===== Hasil Format Data =====")
print(mapped_dataset[0]["text"])


README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

[INFO] Dataset berhasil diformat!

===== Hasil Format Data =====
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Saranlah slogan untuk kampanye daur ulang<|eot_id|><|start_header_id|>assistant<|end_header_id|>

1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."<|eot_id|>


In [4]:
# konfigurasi LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k-proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("[INFO] LoRA Adapter berhasil disuntikkan ke dalam model!")

Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!


Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.6.9 patched 32 layers with 0 QKV layers, 32 O layers and 32 MLP layers.


[INFO] LoRA Adapter berhasil disuntikkan ke dalam model!


In [5]:
# pembagian dataset (Train & Validation)

# membagi dataset jadi 90% training dan 10% evaluasi (validation)
split_dataset = mapped_dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print("[INFO] Dataset berhasil dibagi!")
print(f"[INFO] Jumlah data training: {len(train_dataset)} baris")
print(f"[INFO] Jumlah data evaluasi: {len(eval_dataset)} baris")

[INFO] Dataset berhasil dibagi!
[INFO] Jumlah data training: 44972 baris
[INFO] Jumlah data evaluasi: 4997 baris


In [6]:
# MENGATASI ERROR: Patching Trainer agar menerima tokenizer
import transformers
from transformers import Trainer

# Simpan referensi asli Trainer
original_init = Trainer.__init__

# Buat fungsi init baru yang membuang argumen 'tokenizer' jika dia keberatan
def fixed_init(self, *args, **kwargs):
    kwargs.pop("tokenizer", None) # Memastikan tokenizer tidak dikirim ke super()
    original_init(self, *args, **kwargs)

# Pasang "Patch" ini ke Trainer
Trainer.__init__ = fixed_init

print("[INFO] Patch Trainer berhasil dipasang! Sekarang argumen tokenizer akan diabaikan oleh sistem.")

[INFO] Patch Trainer berhasil dipasang! Sekarang argumen tokenizer akan diabaikan oleh sistem.


In [7]:
# SFT Trainer - Eksperimen 1
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch

torch.cuda.empty_cache()

# ini untuk menentukan argumen pelatihan (Hyperparameters)
training_args_exp1 = TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_steps = 50,
    max_steps = 800,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 50,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs_exp1",
    eval_strategy = "steps",
    eval_steps = 100,
)

# inisialisasi SFTTrainer
trainer_exp1 = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    max_seq_length = 512,
    dataset_text_field = "text",
    packing = True,
    args = training_args_exp1,
)

print("[INFO] Selesai: SFTTrainer Eksperimen 1 sudah siap!")

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

[INFO] Selesai: SFTTrainer Eksperimen 1 sudah siap!


In [8]:
print("[INFO] Memulai proses pelatihan Eksperimen 1. sabar ini agak lama...!")
trainer_stats_exp1 = trainer_exp1.train()
print("[DONE] Pelatihan Eksperimen 1 Selesai!")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 21,575 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 39,321,600 of 8,069,582,848 (0.49% trained)


[INFO] Memulai proses pelatihan Eksperimen 1. sabar ini agak lama...!


`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,1.283010,1.261367
200,1.243736,1.228219
300,1.238739,1.213175
400,1.205074,1.198974
500,1.216290,1.189676
600,1.189066,1.181912


Step,Training Loss,Validation Loss
100,1.283010,1.261367
200,1.243736,1.228219
300,1.238739,1.213175
400,1.205074,1.198974
500,1.216290,1.189676
600,1.189066,1.181912
700,1.188369,1.174284
800,1.170520,1.170425


[DONE] Pelatihan Eksperimen 1 Selesai!


In [ ]:
import gc
import torch

del trainer_exp1
gc.collect()
torch.cuda.empty_cache()

print("[INFO] Memori GPU berhsil dibersihkan! siap untuk ekperimen 2!")

In [ ]:
# SFT Trainer - Eksperimen 2
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch

torch.cuda.empty_cache()

# Konfigurasi Eksperimen 2 (Hyperparameter)
training_args_exp2 = TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_steps = 50,
    max_steps = 800,
    learning_rate = 1e-4,                 # diperlambat jadi 1e-4 agar Belajar lebih pelan & teliti
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 50,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",         # diubah ke Kurva Cosine untuk pengereman halus
    seed = 3407,
    output_dir = "outputs_exp2",          # untuk Folder penyimpanan baru
    eval_strategy = "steps",
    eval_steps = 100,
)

# Inisialisasi SFTTrainer
trainer_exp2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    max_seq_length = 512,
    dataset_text_field = "text",
    packing = True,
    args = training_args_exp2,
)

In [ ]:
print("[INFO] Memulai proses pelatihan Eksperimen 2. sabar ini agak lama...!")
trainer_stats_exp2 = trainer_exp2.train()
print("[DONE] Pelatihan Eksperimen 2 Selesai!")

In [ ]:
from unsloth import FastLanguageModel
import torch

path_checkpoint = "outputs_exp1/checkpoint-800"

print(f"Memuat model dari: {path_checkpoint}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = path_checkpoint,
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

print("[INFO] Model berhasil dimuat ke dalam memori GPU dan siap diunggah!")

In [ ]:
# Push model ke Hugging Face Hub
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")

login(hf_token)

nama_repo_hf = "Saranjana/Llama-3-8b-Legal-RAG"
print(f"[INFO] Sedang mengunggah model ke: {nama_repo_hf}")

model.push_to_hub_merged(
    nama_repo_hf,
    tokenizer,
    save_method = "merged_16bit",
    token = hf_token
)

print("[DONE] Model berhasil di upload ke Hugging Face Hub!")

[INFO] Sedang mengunggah model ke: Saranjana/Llama-3-8b-Legal-RAG


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Saranjana/Llama-3-8b-Legal-RAG/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...


Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [04:13<01:23, 83.62s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:31<00:00, 67.89s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Saranjana/Llama-3-8b-Legal-RAG`
[DONE] Model berhasil di upload ke Hugging Face Hub!
